# LongFlow P1 — 10K batched caching run (overnight)

Runtime: **L4 GPU**. Resumable: cache on Drive, safe to interrupt any time; re-run both cells to continue. Expect ~5h for 10K at batch 8 (measured 1.7s/utt). Drag **`longflow_bundle.zip`** in when the cold start asks.

In [ ]:
# ===== COLD START (idempotent) =====
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import os, sys, time
CACHE_DIR = "/content/drive/MyDrive/longflow_p1_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

assert os.path.exists("/content/longflow_bundle.zip"), "DRAG longflow_bundle.zip IN, then re-run"
!cd /content && unzip -q -o longflow_bundle.zip
sys.path.insert(0, "/content")
from src.cache.capture import BatchedSampleCapture, UtteranceCache, save_utterance
FRAME_ID = processor.tokenizer.convert_tokens_to_ids("<|vision_pad|>")
print("READY")

In [ ]:
# ===== Batched caching loop (resumable) =====
import soundfile as sf
from pathlib import Path
from datasets import load_dataset

TARGET = 10_000
BS = 8
ds = load_dataset("mythicinfinity/libritts_r", "clean", split="train.clean.360", streaming=True)
done = {f.stem for f in Path(CACHE_DIR).glob("*.pt")}
print(f"resuming with {len(done)} already cached")
t0, n0 = time.time(), len(done)

def flush(buf):
    texts = [f"Speaker 1: {b['text']}\n" for b in buf]
    voices = [[b["prompt"]] for b in buf]
    inputs = processor(text=texts, voice_samples=voices, return_tensors="pt", padding=True)
    inputs = {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}
    try:
        with BatchedSampleCapture(model) as bcap, torch.inference_mode():
            out = model.generate(**inputs, tokenizer=processor.tokenizer, cfg_scale=1.3)
        seq = out.sequences if hasattr(out, "sequences") else out
        pl = inputs["input_ids"].shape[1]
        streams = [seq[i, pl:].tolist() for i in range(len(buf))]
        parts = bcap.split_utterances(streams, frame_id=FRAME_ID)
        for b, (hidden, latent) in zip(buf, parts):
            utt = UtteranceCache(utt_id=b["uid"], text=b["text"], hidden=hidden,
                                 latent=latent, meta={"speaker": b["spk"], "batched": True})
            save_utterance(utt, f"{CACHE_DIR}/{b['uid']}.pt")
            done.add(b["uid"])
    except Exception as e:
        print("batch failed (skipped, will not retry this session):", repr(e)[:150])

buf = []
for ex in ds:
    if len(done) >= TARGET:
        break
    uid = str(ex["id"]).replace("/", "_")
    if uid in done:
        continue
    text = ex["text_normalized"].strip()
    audio, sr = ex["audio"]["array"], ex["audio"]["sampling_rate"]
    if not (30 <= len(text) <= 180) or len(audio) < 3 * sr:
        continue
    p = f"/content/_bp{len(buf)}.wav"
    sf.write(p, audio[: 3 * sr], sr)
    buf.append({"uid": uid, "text": text, "prompt": p, "spk": str(ex.get("speaker_id"))})
    if len(buf) == BS:
        flush(buf)
        buf = []
        if (len(done) - n0) % 200 < BS and len(done) > n0:
            r = (time.time() - t0) / max(1, len(done) - n0)
            print(f"{len(done)}/{TARGET}  {r:.1f}s/utt  ETA {(TARGET-len(done))*r/3600:.1f}h")
print(f"DONE: {len(done)} cached")